In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
cd ..

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import List, Dict
import re
from pypdf import PdfReader
import docx
from docx import Document as DocxDocument
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

_OCR_AVAILABLE = True
OCR_LANG = os.getenv("OCR_LANG", "fra")
OCR_DPI = int(os.getenv("OCR_DPI", "300"))
OCR_MIN_ALNUM_RATIO = 0.05
OCR_MIN_CHARS = 30
USE_OCR_FALLBACK =True


def normalize_text(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in s.split("\n")]
    out_lines, empty = [], False
    for ln in lines:
        if ln == "":
            if not empty:
                out_lines.append("")
            empty = True
        else:
            out_lines.append(ln)
            empty = False
    return "\n".join(out_lines).strip()


def _is_poor_text(s: str) -> bool:
    if not s:
        return True
    if len(s) < OCR_MIN_CHARS:
        return True
    alnum = sum(1 for c in s if c.isalnum())
    ratio = alnum / max(1, len(s))
    return ratio < OCR_MIN_ALNUM_RATIO


def _ocr_page(path: str, page_no: int) -> str:
    """Run OCR for a single page (1-indexed page_no). Returns extracted text or empty string on failure."""
    if not _OCR_AVAILABLE:
        return ""
    try:
        imgs = convert_from_path(path, dpi=OCR_DPI, first_page=page_no, last_page=page_no, fmt='jpeg')
        if not imgs:
            return ""
        img = imgs[0]
        try:
            txt = pytesseract.image_to_string(img, lang=OCR_LANG)
        except Exception:
            # fallback without lang
            txt = pytesseract.image_to_string(img)
        return txt or ""
    except Exception as e:
        print(f"OCR error on {path} page {page_no}: {e}")
        return ""


def read_pdf_to_text(path: str) -> str:
    reader = PdfReader(path)
    parts: List[str] = []
    for i, page in enumerate(reader.pages, start=1):
        txt = (page.extract_text() or "").strip()
        # If extraction seems poor, try OCR for that page
        if _is_poor_text(txt) and USE_OCR_FALLBACK:
            ocr_txt = _ocr_page(path, i)
            if _is_poor_text(ocr_txt):
                # keep original (even if empty) but log that OCR couldn't help
                if ocr_txt:
                    txt = ocr_txt.strip()
                else:
                    print(f"[WARN] Page {i} of {Path(path).name} yielded poor text and OCR failed or missing dependencies.")
            else:
                txt = ocr_txt.strip()
        if txt:
            parts.append(f"[PAGE {i}]\n{txt}")
    return normalize_text("\n\n".join(parts))


def read_docx_to_text(path: str) -> str:
    doc = DocxDocument(path)
    parts: List[str] = []
    # paragraphs
    for p in doc.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    # tables -> simple row text
    for tbl in doc.tables:
        for row in tbl.rows:
            cells = [re.sub(r"\s+", " ", (c.text or "").strip()) for c in row.cells]
            cells = [c for c in cells if c]
            if cells:
                parts.append(" | ".join(cells))
    return normalize_text("\n".join(parts))

# Collect inputs from ./data/in recursively (preserve subpaths when writing to out)
BASE_IN = Path(os.getenv("SP_BASE_IN", "./data/in/conges"))
BASE_FALLBACK = Path(os.getenv("SP_BASE_FALLBACK", "./data"))
BASE_OUT = Path(os.getenv("SP_BASE_OUT", "./data/out/conges"))

# Gather supported files
inputs = []
if BASE_IN.exists():
    inputs.extend(sorted([str(p) for p in BASE_IN.rglob("*.pdf")]))
    inputs.extend(sorted([str(p) for p in BASE_IN.rglob("*.docx")]))
else:
    # backward-compatible: look under ./data for files if ./data/in doesn't exist
    inputs.extend(sorted([str(p) for p in BASE_FALLBACK.rglob("*.pdf")]))
    inputs.extend(sorted([str(p) for p in BASE_FALLBACK.rglob("*.docx")]))

# remove duplicates while keeping order
seen = set()
inputs_unique = []
for p in inputs:
    if p not in seen:
        seen.add(p)
        inputs_unique.append(p)
inputs = inputs_unique

results: List[Dict[str, str]] = []
for p in inputs:
    pp = Path(p)
    ext = pp.suffix.lower()
    if ext == ".pdf":
        text = read_pdf_to_text(p)
    elif ext == ".docx":
        text = read_docx_to_text(p)
    else:
        print(f"Skipping unsupported file: {p}")
        continue

    # compute relative path under BASE_IN if possible, else under BASE_FALLBACK, else use filename
    try:
        rel = pp.relative_to(BASE_IN)
    except Exception:
        try:
            rel = pp.relative_to(BASE_FALLBACK)
        except Exception:
            rel = Path(pp.name)

    out = BASE_OUT / rel
    out = out.with_suffix('.txt')
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    rel_out = out.relative_to(BASE_OUT)
    print(f"✓ {pp} → {rel_out}  ({len(text)} chars, {text.count('\n') + (1 if text else 0)} lines)")

# If OCR dependencies are missing and OCR fallback is enabled, print helpful instructions
if USE_OCR_FALLBACK and not _OCR_AVAILABLE:
    print("\n[INFO] OCR fallback was enabled but 'pytesseract' and/or 'pdf2image' are not installed or poppler is missing.")
    print("Install with (macOS):\n  brew install poppler tesseract\n  pip install pytesseract pdf2image pillow")


In [ ]:
# Q/R-aware chunking for French HR texts (questions, sub-questions, tables)
# ------------------------------------------------------------------------
# How to use:
# 1) Put your cleaned .txt files on disk.
# 2) Edit INPUT_FILES and/or PATTERNS below.
# 3) Run. You'll get:
#    - chunks_qna.jsonl (one chunk per line, ready for embeddings)
#    - a printed summary and a small preview
#
# Chunk roles:
#   - Q_ONLY       : the question alone
#   - QA_COMPOSITE : "Q: ...\n\nR: ..." (compact joined context)
#   - A_ATOMIC     : answer segments (~1200 chars with 200 overlap), prefixed by a short Q
#   - TABLE        : paragraphs that look like a table (Markdown-like or "Tableau ...")

from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Optional
from glob import glob

import re, hashlib, json
import pandas as pd

# -----------------------------
# 0) Configure your input files
# -----------------------------
# You can either list explicit files in INPUT_FILES or use PATTERNS (recommended)
INPUT_FILES: List[str] = [p.strip() for p in os.getenv("SP_INPUT_FILES", "").split(",") if p.strip()]
PATTERNS: List[str] = [p.strip() for p in os.getenv("SP_INPUT_PATTERNS", "./data/out/conges/*.txt").split(",") if p.strip()]
OUT_JSONL = os.getenv("SP_CHUNKS_JSONL", "./data/out/chunked/chunks_qna_sp.jsonl")

# -----------------------------
# 1) Utilities
# -----------------------------
def norm(s: str) -> str:
    """Normalize line endings, trim lines, collapse 3+ blank lines to 2."""
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join(line.strip() for line in s.split("\n"))
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def sha1_u(s: str) -> str:
    """Stable lowercase SHA1 hex of a string."""
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def expand_inputs(files: List[str], patterns: List[str]) -> List[str]:
    """Return a deduplicated, ordered list of file paths from explicit files and glob patterns."""
    out: List[str] = []
    seen = set()
    for f in files or []:
        if f and f not in seen:
            seen.add(f); out.append(f)
    for pat in patterns or []:
        for p in sorted(glob(pat, recursive=True)):
            if p not in seen:
                seen.add(p); out.append(p)
    return out

def compute_thematique(path: str) -> str:
    """
    Return the folder path under data/in|data/out, excluding the filename.
    Examples:
      ./data/out/licenciement/file.txt            -> 'licenciement'
      ./data/out/licenciement                     -> 'licenciement'   (directory case)
      /abs/.../data/out/recrutement/typ/file.txt  -> 'recrutement/typ'
      ./data/out/file.txt                         -> ''               (direct child)
    """
    try:
        parts = list(Path(path).parts)
    except Exception:
        parts = str(path).replace("\\", "/").split("/")

    # drop leading '.' / '' noise
    parts = [pp for pp in parts if pp not in ('.', '', None)]
    lower = [pp.lower() for pp in parts]

    # cut everything up to and including 'data'
    if 'data' in lower:
        i = lower.index('data')
        tail = parts[i+1:]
    else:
        tail = parts[:]  # fallback; still try to interpret in/out below

    # remove leading 'in'/'out' if present
    if tail and tail[0].lower() in ('in', 'out'):
        tail = tail[1:]

    # If last segment looks like a filename, drop it; if path points to a directory, keep it.
    if tail:
        last = tail[-1]
        looks_like_file = ('.' in last[1:]) and (not last.startswith('.'))
        if looks_like_file and len(tail) >= 2:
            tail = tail[:-1]

    them = '/'.join(tail).strip('/')
    return them

# -----------------------------
# 2) Heuristics (FR)
# -----------------------------
# Lines that look like questions or Q-headings commonly found in French HR docs
Q_PAT = re.compile(
    r"^(?:Q(?:uestion)?\s*[:\-]\s*|"
    r"(?:Quel(?:le|s)?|Comment|Pourquoi|Quand|Dans quel(?:le)?|L'agent|Le contractuel|Vous)\b.*\?\s*$|"
    r"(?:Quelle est la procédure|Le contractuel a-t-il droit|L'agent a-t-il droit)\b.*)",
    re.IGNORECASE,
)

# Headings: roman numerals, numbered items, or "FICHE n"
HEAD_PAT = re.compile(r"^[IVXLC]+[\.\-]\s+|^\d+[\.\)]\s+|^(FICHE\s*\d+)$", re.IGNORECASE)

# Tables (Markdown-like lines with pipes or lines that start with "Tableau")
TABLE_HINT = re.compile(r"^(Tableau\b|.*\|.*\|.*)$", re.IGNORECASE)

# -----------------------------
# 3) Data model
# -----------------------------
@dataclass
class Chunk:
    qa_id: str
    parent_qa_id: Optional[str]
    role: str              # Q_ONLY / QA_COMPOSITE / A_ATOMIC / TABLE
    section_path: str
    chunk_index: int
    text: str
    source_name: str
    lang: str = "fr"
    thematique: str = ""

# -----------------------------
# 4) Block parsing (Q -> A) with section tracking & smart parentage
# -----------------------------
def parse_qna_blocks(text: str) -> List[Dict]:
    """
    Produce blocks: {"qa_id","parent_qa_id","section_path","question","answer"}.
    A question-like line starts a block; subsequent lines belong to its answer
    until the next question or a heading. Sub-questions attach to the latest
    root question within the same section.
    """
    lines = text.split("\n")
    section_stack: List[str] = []
    blocks: List[Dict] = []
    current_q: Optional[str] = None
    current_ans: List[str] = []
    section_path = ""

    # Track the most recent root question per section
    last_root: Dict[str, str] = {}

    SUBQ = re.compile(
        r"(procédure|préavis|montant|conditions|conséquences|droit|indemnité|reclassement|"
        r"cas particulier|délais|pièces|modalités|exceptions?)",
        re.IGNORECASE
    )

    def flush():
        nonlocal current_q, current_ans, section_path
        if current_q is None:
            return
        ans = "\n".join(current_ans).strip()
        qnorm = re.sub(r"\s+", " ", current_q).strip().lower()
        qa_id = sha1_u(qnorm)

        is_sub = bool(SUBQ.search(current_q or ""))
        parent_qa_id = None
        if is_sub and section_path in last_root:
            parent_qa_id = last_root[section_path]
        else:
            # become / refresh the root for this section
            last_root[section_path] = qa_id

        blocks.append({
            "qa_id": qa_id,
            "parent_qa_id": parent_qa_id,
            "section_path": section_path,
            "question": (current_q or "").strip(),
            "answer": ans,
        })
        current_q, current_ans = None, []

    for raw in lines:
        ln = raw.strip()

        if not ln:
            if current_q is not None:
                current_ans.append("")  # paragraph boundary
            continue

        # Section/heading (don’t steal lines that are also questions)
        if HEAD_PAT.match(ln) and not Q_PAT.match(ln):
            title = re.sub(r"^[IVXLC]+[\.\-]\s+|\d+[\.\)]\s+","", ln).strip()
            if title:
                section_stack.append(title)
                section_stack[:] = section_stack[-4:]    # keep last 4 levels
                section_path = " > ".join(section_stack)
            continue

        if Q_PAT.match(ln):
            flush()
            current_q = ln
            current_ans = []
        else:
            if current_q is not None:
                current_ans.append(ln)

    flush()
    return blocks

# -----------------------------
# 5) Answer-aware chunking
# -----------------------------
def hard_wrap(text: str, max_chars: int, overlap: int) -> List[str]:
    if max_chars <= 0:
        return [text]
    res, i, n = [], 0, len(text)
    step = max(1, max_chars - overlap)
    while i < n:
        res.append(text[i:i+max_chars])
        i += step
    return res

def split_on_paragraphs(text: str, max_chars: int, overlap: int) -> List[str]:
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    out: List[str] = []
    buf = ""
    for p in paras:
        extra = (2 if buf else 0)
        if len(buf) + extra + len(p) <= max_chars:
            buf = (buf + "\n\n" + p) if buf else p
        else:
            if buf:
                out.append(buf)
            if len(p) > max_chars:
                out.extend(hard_wrap(p, max_chars, overlap)); buf = ""
            else:
                buf = p
    if buf:
        out.append(buf)

    final: List[str] = []
    for c in out:
        if len(c) <= max_chars:
            final.append(c)
        else:
            final.extend(hard_wrap(c, max_chars, overlap))
    return final

def make_chunks(blocks: List[Dict], source_name: str,
                max_chars: int = 1200, overlap: int = 200) -> List[Chunk]:
    rows: List[Chunk] = []
    for b in blocks:
        q = (b["question"] or "").strip()
        a = (b["answer"] or "").strip()
        qa_id = b["qa_id"]
        parent = b.get("parent_qa_id")  # root if sub-question, else None
        section = b["section_path"]

        # Q_ONLY: if sub-question, point Q to the root; else parent=None
        if q:
            rows.append(Chunk(qa_id, parent, "Q_ONLY", section, 0, q, source_name))

        # Children always hang under the question; if the question is a child,
        # they inherit the root (parent or qa_id).
        parent_for_children = parent or qa_id

        next_idx = 1  # 1 reserved for QA_COMPOSITE
        if a:
            composite = f"Q: {q}\n\nR: {a}" if q else a
            rows.append(Chunk(qa_id, parent_for_children, "QA_COMPOSITE", section, 1, composite[:1500], source_name))
            next_idx = 2

            # A_ATOMIC chunks
            for k, ch in enumerate(split_on_paragraphs(a, max_chars, overlap), start=next_idx):
                q_short = re.sub(r"\s+", " ", q)[:160]
                rows.append(Chunk(qa_id, parent_for_children, "A_ATOMIC", section, k, f"Q: {q_short}\nR: {ch}", source_name))
            next_idx = k + 1 if 'k' in locals() else next_idx

            # TABLE chunks
            for para in re.split(r"\n{2,}", a):
                p = (para or "").strip()
                if p and TABLE_HINT.match(p):
                    rows.append(Chunk(qa_id, parent_for_children, "TABLE", section, next_idx, p, source_name))
                    next_idx += 1

    return rows

# -----------------------------
# 6) Process files
# -----------------------------
def process_files(paths: List[str], out_jsonl: str) -> pd.DataFrame:
    all_chunks: List[Chunk] = []
    for p in paths:
        src = Path(p)
        if not src.exists():
            print(f"Skipping (not found): {p}")
            continue
        text = norm(src.read_text(encoding="utf-8", errors="ignore"))
        blocks = parse_qna_blocks(text)
        chunks = make_chunks(blocks, source_name=src.name)
        them = compute_thematique(str(src))
        for c in chunks:
            c.thematique = them
        all_chunks.extend(chunks)
        print(f"✓ {src.name}: {len(blocks)} Q-blocks → {len(chunks)} chunks (thematique='{them}')")

    if not all_chunks:
        print("No chunks produced.")
        return pd.DataFrame(columns=[
            "role","section_path","qa_id","parent_qa_id","chunk_index","text","source_name","thematique"
        ])

    outp = Path(out_jsonl)
    outp.parent.mkdir(parents=True, exist_ok=True)
    with outp.open("w", encoding="utf-8") as f:
        for c in all_chunks:
            f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

    print(f"→ JSONL written: {outp} ({len(all_chunks)} rows)")
    df = pd.DataFrame([asdict(c) for c in all_chunks])
    return df

# -----------------------------
# 7) Run
# -----------------------------
if __name__ == "__main__":
    ALL_INPUTS = expand_inputs(INPUT_FILES, PATTERNS)
    print(f"Found {len(ALL_INPUTS)} input .txt files (sample={ALL_INPUTS[:3] if ALL_INPUTS else []})")

    df_preview = process_files(ALL_INPUTS, OUT_JSONL)

    # Compact preview (avoid giant dumps in terminals)
    cols = ["role","section_path","qa_id","parent_qa_id","chunk_index","source_name","thematique","text"]
    with pd.option_context("display.max_colwidth", 140):
        print(df_preview[cols].head(25).to_string(index=False))



In [ ]:
from sentence_transformers import SentenceTransformer
import torch, numpy as np
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import fastparquet  # noqa
engine = "fastparquet"

MODEL_NAME = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

BATCH_SIZE = 64                  # adjust to RAM/VRAM
NORMALIZE  = True                # cosine-ready vectors
EMBED_COL  = os.getenv("EMBEDDING_COLUMN", "embedding_m3")



device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {device}")

def format_passage(text: str) -> str:
    """Model-aware passage formatting."""
    if MODEL_NAME.startswith("intfloat/multilingual-e5"):
        return f"passage: {text or ''}"
    # bge-m3 / gte-multilingual-base: no instruction needed
    return text or ""

# ==========================================================
# Fixed & improved embeddings writer (robust, model-aware)
# ==========================================================


# ---------- Config / defaults ----------
IN_JSONL    = os.getenv("SP_CHUNKS_JSONL", "./data/out/chunked/chunks_qna_sp.jsonl")
TAG         = (MODEL_NAME.replace("/", "_").replace("-", "_")).lower()
OUT_PARQUET = os.getenv("SP_EMB_OUT_PARQUET", f"./data/out/chunks_{TAG}_SP.parquet")
OUT_NPY     = os.getenv("SP_EMB_OUT_NPY", f"./data/out/embeddings_{TAG}_SP.npy")
OUT_JSONL   = os.getenv("SP_EMB_OUT_JSONL", f"./data/out/chunks_{TAG}_with_emb_SP.jsonl")

# ---------- Helpers ----------
def sha1_u(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def make_hash_id(row) -> str:
    key = f"{row.get('source_name','')}|{row.get('qa_id','')}|{row.get('role','')}|{row.get('chunk_index',0)}|{(row.get('text','')[:256])}"
    return sha1_u(key)

# ---------- Load chunks ----------
rows = []
with open(IN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
assert not df.empty, f"Le fichier {IN_JSONL} est vide."

# Stable ids
df["hash_id"] = df.apply(make_hash_id, axis=1)

# ---------- Build corpus (model-aware) ----------
corpus = [format_passage(t) for t in df["text"].astype(str).tolist()]

# ---------- Encode in batches ----------
vecs_all = []
for i in range(0, len(corpus), BATCH_SIZE):
    batch = corpus[i:i+BATCH_SIZE]
    vecs = model.encode(
        batch,
        batch_size=len(batch),           # respect small leftover batch
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE,
        show_progress_bar=True,
    )
    vecs_all.append(vecs)
emb = np.vstack(vecs_all).astype(np.float32)  # (N, d)

# attach to df
df[EMBED_COL] = [v.tolist() for v in emb]
df['source']="SERVICE PUBLIC"
# ---------- Save Parquet (embeddings as JSON strings) ----------
Path(OUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
df_parquet = df.copy()
df_parquet[EMBED_COL] = df_parquet[EMBED_COL].apply(json.dumps)  # store as JSON string




   
if engine is None:
    print("⚠️ Neither 'fastparquet' nor 'pyarrow' found. Skipping Parquet save.")
else:
    df_parquet.to_parquet(OUT_PARQUET, index=False, engine=engine)
    print(f"Parquet saved to: {OUT_PARQUET} (engine={engine}, embeddings as JSON strings)")

# ---------- Save NPY matrix ----------
np.save(OUT_NPY, emb)
print("NPY matrix saved to:", OUT_NPY, emb.shape)

# ---------- Save JSONL with embeddings inline ----------
Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("JSONL+emb saved to:", OUT_JSONL)

# ---------- Peek ----------
print(df[["source_name","role","section_path","chunk_index","hash_id"]].head(10).to_string(index=False))
